# 查看传给 AI Agent 的输入

输入 BV 号 → 爬取评论 → 展示给 DeepSeek 的完整请求内容

## 1. 输入 BV 号并爬取

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import asyncio
from src.config import Config
from src.service import CrawlerService

BV_ID = "BV1rYCvYJEWQ"   # ← 修改为你自己的 BV 号

config = Config.from_env()
config.delay_base = 0.0      # notebook 里不限速
config.delay_jitter = 0.0
svc = CrawlerService(config)

print(f"正在爬取: {BV_ID}")
result = await svc.crawl(BV_ID)
print(f"\n✅ 爬取完成: {result.video_title}")
print(f"   一级评论: {result.total_count} 条")
print(f"   耗时: {result.crawl_time:.1f}s")
if result.errors:
    print(f"   错误: {result.errors}")

## 2. 展平评论 + 构建 Agent 输入

In [ ]:
# 展平树形结构（一级 + 楼中楼全量）
all_comments = []
def walk(comments):
    for c in comments:
        all_comments.append({
            "rpid": str(c.rpid),
            "content": c.content,
            "parent_id": str(c.parent_id or ""),
        })
        if c.replies:
            walk(c.replies)
walk(result.comments)

print(f"总共 {len(all_comments)} 条评论（含楼中楼）")
print(f"每批 {50} 条，分 {(len(all_comments) + 49) // 50} 批发送")

## 3. 查看 System Prompt 结构

In [ ]:
from src.agent.ad_detector import build_system_prompt

system_prompt = build_system_prompt("prompts")
print(f"System Prompt 总长度: {len(system_prompt)} 字符")
print(f"\n=== 前 300 字 ===")
print(system_prompt[:300])
print(f"\n...（中间是 CSV 广告样本 + 正常评论示例）...")
print(f"\n=== 后 300 字 ===")
print(system_prompt[-300:])

## 4. 查看第一批的 User Message（给 DeepSeek 的实际输入）

In [ ]:
import json

batch = all_comments[:50]  # 第一批 50 条

# 这就是发给 DeepSeek 的 user message
user_message = json.dumps(batch, ensure_ascii=False, indent=2)

print(f"=== 第 1 批 User Message ({len(batch)} 条评论) ===")
print(f"JSON 长度: {len(user_message)} 字符\n")
print(user_message[:2000])
if len(user_message) > 2000:
    print(f"\n... 省略 {len(user_message) - 2000} 字符 ...")

## 5. 查看完整请求结构（System + User）

In [ ]:
print("="*60)
print("实际 API 调用结构")
print("="*60)
print(f"""
POST https://api.deepseek.com/v1/chat/completions
  model: {config.deepseek_model}
  temperature: 0.1
  max_tokens: 4096
  response_format: {{"type": "json_object"}}
  messages: [
    {{
      role: "system",
      content: "{system_prompt[:80]}..."  ({len(system_prompt)} chars)
    }},
    {{
      role: "user",
      content: "{user_message[:80]}..."  ({len(user_message)} chars, {len(batch)} 条评论)
    }}
  ]
""")

## 6. 查看期望的返回格式

In [ ]:
print("期望 DeepSeek 返回的 JSON 结构：")
print()

example_output = {
    "judgments": [
        {
            "rpid": str(all_comments[0]["rpid"]) if all_comments else "123456",
            "is_ad": False,
            "ad_type": None,
            "reason": "正常评论，讨论视频内容"
        },
        {
            "rpid": str(all_comments[1]["rpid"]) if len(all_comments) > 1 else "789012",
            "is_ad": True,
            "ad_type": "引流",
            "reason": "引导加微信，疑似广告引流"
        }
    ]
}

print(json.dumps(example_output, ensure_ascii=False, indent=2))

## 7. 逐批预览（查看每一批的内容概览）

In [ ]:
batch_size = 50
total_batches = (len(all_comments) + batch_size - 1) // batch_size

for batch_idx in range(min(total_batches, 10)):  # 最多显示前 10 批
    start = batch_idx * batch_size
    end = min(start + batch_size, len(all_comments))
    batch = all_comments[start:end]
    
    print(f"\n--- 第 {batch_idx+1}/{total_batches} 批 ({len(batch)} 条) ---")
    for i, c in enumerate(batch[:3]):  # 每批只显示前 3 条
        content_preview = c["content"][:60]
        parent_info = f" | 回复: {c['parent_id']}" if c["parent_id"] else ""
        print(f"  [{c['rpid']}] {content_preview}…{parent_info}")
    if len(batch) > 3:
        print(f"  ... 还有 {len(batch)-3} 条")

if total_batches > 10:
    print(f"\n... 还有 {total_batches - 10} 批未显示")